In [0]:
from pyspark.sql.functions import col, sum

data_path = "/Volumes/workspace/default/taxi_data/*.parquet"

df_raw = spark.read.parquet(data_path)

print("Schema:")
df_raw.printSchema()

print("Total Rows:", df_raw.count())

initial_partitions = df_raw.selectExpr("spark_partition_id() as pid") \
                           .distinct() \
                           .count()

print("Initial Partitions:", initial_partitions)

df_raw = df_raw.repartition(200)

new_partitions = df_raw.selectExpr("spark_partition_id() as pid") \
                       .distinct() \
                       .count()

print("Partitions After Repartition:", new_partitions)

null_counts = df_raw.select([
    sum(col(c).isNull().cast("int")).alias(c) for c in df_raw.columns
])

null_counts.show(truncate=False)

# Save Bronze layer
df_raw.write.mode("overwrite").parquet(
    "/Volumes/workspace/default/taxi_data/bronze_2019"
)

print("Bronze layer saved successfully.")